In [ ]:
import os; os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import sys, site, subprocess, shutil, glob

# Fix ptxas-blackwell permission
PTXAS_SRC = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
PTXAS_DST = '/kaggle/working/ptxas-blackwell'
if os.path.exists(PTXAS_SRC):
    shutil.copy2(PTXAS_SRC, PTXAS_DST)
    os.chmod(PTXAS_DST, 0o755)
    _OrigPopen = subprocess.Popen
    class PatchedPopen(_OrigPopen):
        def __init__(self, args, *a, **kw):
            if isinstance(args, (list, tuple)):
                args = list(args)
                if args and 'ptxas-blackwell' in str(args[0]):
                    args[0] = PTXAS_DST
            super().__init__(args, *a, **kw)
    subprocess.Popen = PatchedPopen
    print(f'ptxas fix applied')

for base in ['/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script',
             '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script']:
    if os.path.exists(base):
        site.addsitedir(base)
        for pp in glob.glob(os.path.join(base, '**/python_packages'), recursive=True):
            if pp not in sys.path: site.addsitedir(pp)
print('Setup complete')

In [ ]:
import json, statistics, torch
from pathlib import Path
from collections import Counter
import kagglehub, mamba_ssm
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

OUTPUT_DIR = Path('/kaggle/working')
MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, device_map='auto', trust_remote_code=True, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Model loaded. Vocab: {len(tokenizer)}')

lora_config = LoraConfig(r=32, lora_alpha=64, target_modules=r'.*\.(in_proj|out_proj|up_proj|down_proj)$',
                         lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
COT_PATH = None
for p in glob.glob('/kaggle/input/**/cot_train.jsonl', recursive=True):
    COT_PATH = p; break
if not COT_PATH:
    for d in sorted(Path('/kaggle/input').iterdir()):
        print(f'  {d.name}/', [f.name for f in list(d.iterdir())[:5]])
    raise FileNotFoundError('cot_train.jsonl not found')
examples = [json.loads(l) for l in open(COT_PATH) if l.strip()]
print(f'Loaded {len(examples)} from {COT_PATH}')
texts = [tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False) for ex in examples]
print(f'Token lengths: mean={statistics.mean([len(tokenizer.encode(t)) for t in texts[:200]]):.0f}')

In [ ]:
from torch.utils.data import Dataset, random_split

class SFTDataset(Dataset):
    def __init__(self, texts, tok, max_len):
        self.items = []
        for t in texts:
            enc = tok(t, truncation=True, max_length=max_len, padding='max_length', return_tensors='pt')
            self.items.append({k: v.squeeze(0) for k, v in enc.items()})
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        item = {k: v.clone() for k, v in self.items[i].items()}
        item['labels'] = item['input_ids'].clone()
        item['labels'][item['labels'] == tokenizer.pad_token_id] = -100
        return item

ds = SFTDataset(texts, tokenizer, 2048)
train_ds, val_ds = random_split(ds, [len(ds)-max(1,len(ds)//20), max(1,len(ds)//20)], generator=torch.Generator().manual_seed(42))
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR/'ckpt'), num_train_epochs=3,
    per_device_train_batch_size=1, gradient_accumulation_steps=8,
    learning_rate=2e-5, lr_scheduler_type='cosine', warmup_ratio=0.05,
    weight_decay=0.01, bf16=True, logging_steps=10,
    eval_strategy='no',
    save_strategy='epoch', save_total_limit=1,
    gradient_checkpointing=True,
    optim='adamw_torch_fused', report_to='none', max_grad_norm=1.0,
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds)
print('Starting training...')
trainer.train()
print('Training complete!')

In [ ]:
model.save_pretrained(str(OUTPUT_DIR))
os.chdir(str(OUTPUT_DIR))
subprocess.run(['zip', 'submission.zip', 'adapter_config.json', 'adapter_model.safetensors'], check=True)
print(f'submission.zip: {os.path.getsize("submission.zip")/1024/1024:.1f} MB')
print('Done!')